# 예제 02. 이미지를 MLP 입력으로 바꾸기
빅데이터프로그래밍 · 7주차

## 목표
- 2차원 이미지를 1차원 벡터로 펼친다
- `view` `reshape` `nn.Flatten` 세 방법을 비교한다
- batch 차원은 남겨야 한다는 규칙을 확인한다

MLP는 2차원 이미지를 그대로 받지 못합니다. 28×28을 784개의 숫자로 펼쳐 넣습니다.


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_set = datasets.MNIST(root="./data", train=True, download=True,
                           transform=transforms.ToTensor())
loader = DataLoader(train_set, batch_size=64, shuffle=True)


## 1. batch 하나의 모양


In [ ]:
images, labels = next(iter(loader))

print("images:", images.shape)    # (64, 1, 28, 28)
print("labels:", labels.shape)    # (64,)
print("한 장  :", images[0].shape)


`(batch, 채널, 높이, 너비)` 입니다. 4차원입니다.


## 2. 펼치기 — batch 차원은 남깁니다
28 × 28 = 784. batch 64개는 그대로 두고 뒤만 합칩니다.


In [ ]:
flat1 = images.view(images.shape[0], -1)          # -1 은 알아서 계산
flat2 = images.reshape(64, 784)                   # 숫자를 직접 씀
flat3 = nn.Flatten()(images)                      # 층으로

print("view    :", flat1.shape)
print("reshape :", flat2.shape)
print("Flatten :", flat3.shape)
print("세 결과가 같은가:", torch.equal(flat1, flat2) and torch.equal(flat2, flat3))


## 3. batch 차원까지 합치면 — 흔한 실수


In [ ]:
wrong = images.view(-1)          # 전부 한 줄로
print("잘못된 모양:", wrong.shape)      # (50176,) = 64 × 784
print("→ 데이터 64개가 하나로 뭉쳐졌습니다")

right = images.view(images.shape[0], -1)
print("올바른 모양:", right.shape)


## 4. 모델 안에 Flatten을 넣기
`nn.Flatten` 을 첫 층으로 두면 4차원 이미지를 그대로 넣을 수 있습니다.


In [ ]:
model = nn.Sequential(
    nn.Flatten(),              # (64,1,28,28) → (64,784)
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

print(model)
print("\n출력:", model(images).shape)     # (64, 10)


In [ ]:
# 층마다 shape 확인
h = images
for i, layer in enumerate(model):
    h = layer(h)
    print(f"{i} {layer.__class__.__name__:8s} → {tuple(h.shape)}")


## 5. Flatten을 빼면 나는 오류
`nn.Linear` 는 마지막 축만 보므로, 28을 784로 알아듣지 못합니다.


In [ ]:
no_flatten = nn.Sequential(nn.Linear(784, 10))
try:
    no_flatten(images)
except RuntimeError as err:
    print("RuntimeError:", err)


## 6. 펼치면 무엇을 잃는가
이웃한 픽셀이 서로 붙어 있다는 정보가 사라집니다. 9주차 CNN이 이 문제를 다룹니다.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].imshow(images[0].squeeze(), cmap="gray"); ax[0].set_title("28 x 28"); ax[0].axis("off")
ax[1].imshow(images[0].view(1, -1), cmap="gray", aspect="auto"); ax[1].set_title("784 x 1 로 펼친 것"); ax[1].axis("off")
plt.tight_layout(); plt.show()


## 직접 해보기
1. batch_size를 32로 바꾸면 펼친 결과의 shape은 얼마인가요?
2. 컬러 이미지 (3, 32, 32) 를 펼치면 몇 개의 숫자가 되나요? 코드로 확인하세요.


In [ ]:
# 여기에 작성하세요
